### 3.4 Дополнительные проверки и выбор моделей

**Входы**

- train и вопросы после попарного анализа

**Выходы**

- Дополнительные наблюдения
- Модели для сравнения и правила выбора

**Что делаем**

Посмотрим на сочетания нескольких признаков, затем выберем модели, которые могут учесть найденные связи.

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


#### Проверим оставшиеся вопросы

Сравним доли дефолтов в группах: например, при разных сочетаниях дохода и суммы кредита или внутри одинаковых грейдов. Рядом показываем число записей: в маленькой группе доля может сильно измениться даже из-за одного случая.

Доход и сумму делим примерно на четыре равные по числу записей группы; ставки внутри грейда — на три, от низких к высоким, с номерами 0–2. Высокими доходами считаем верхние 5% train. Границы всех групп определяем по train.

Также посмотрим на порядок строк исходного файла. Номер строки помогает проверить, как устроены данные, но признаком модели не станет.

Эти сравнения описывают наши данные. Сами по себе они не доказывают, что различия сохранятся в другой выборке или что найден их источник.

In [ ]:
# dev-only: загрузчик Python-кода; в сборке заменяется содержимым файла
exec(compile(__import__('pathlib').Path('src/notebooks/03_analysis/_code/multivariate.py').read_text(encoding='utf-8'), 'src/notebooks/03_analysis/_code/multivariate.py', 'exec'), globals())

#### Какие модели попробуем

| Модель | Основная идея |
| --- | --- |
| Dummy | Одинаковый прогноз для всех заявок — доля дефолтов в train |
| Логистическая регрессия | Складывает взвешенные вклады признаков и переводит результат в оценку риска |
| Случайный лес | Объединяет много деревьев решений. Дерево — цепочка проверок вроде «доход выше такой-то границы?» |
| Градиентный бустинг на деревьях | Добавляет деревья постепенно: каждое уточняет общий результат |

Так сравним простую модель с вариантами, которые могут учитывать более сложные связи и сочетания признаков.

Оставляем все 11 признаков. Связь двух столбцов ещё не означает, что один из них бесполезен для прогноза. Грейд и ставку сохраняем: по нашей постановке они известны до прогноза.

Обучаем на train, выбираем модель по AP на validation, затем там же выбираем порог по F2:

$$F_2=\frac{5\,\mathrm{Precision}\,\mathrm{Recall}}{4\,\mathrm{Precision}+\mathrm{Recall}}.$$

При равной F2 берём больший порог. После этого решение не меняем и проверяем его на test. На train + validation заново не обучаем, чтобы не менять прогнозы, для которых уже выбрали порог.

**Ограничение:** общий EDA и начальная очистка сделаны до разбиения. Поэтому test нельзя считать полностью невиданными данными. При этом обработка пропусков, обучение, выбор модели и порога не используют test. Случайное разбиение также не проверяет работу на будущем периоде или на данных другого банка.

In [ ]:
# dev-only: загрузчик Python-кода; в сборке заменяется содержимым файла
exec(compile(__import__('pathlib').Path('src/notebooks/03_analysis/_code/conclusions.py').read_text(encoding='utf-8'), 'src/notebooks/03_analysis/_code/conclusions.py', 'exec'), globals())